In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import torch
import gc

In [3]:
# clearing GPU cache:
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [4]:
# force garbage collection:
gc.collect()
print('cache cleared and garbage collected!')

cache cleared and garbage collected!


In [5]:
proj_path = "/content/drive/MyDrive/llm_from_scratch/src"
data_path = "/content/drive/MyDrive/llm_from_scratch/datasets"

In [6]:
import os, sys
sys.path.append(proj_path)
sys.path.append(data_path)
os.chdir(proj_path)
print(os.getcwd())

/content/drive/MyDrive/llm_from_scratch/src


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [8]:
from loading_weights import gpt as model
print("model loaded successfully.")

File already exists and is up-to-date: gpt2/774M/checkpoint
File already exists and is up-to-date: gpt2/774M/encoder.json
File already exists and is up-to-date: gpt2/774M/hparams.json
File already exists and is up-to-date: gpt2/774M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2/774M/model.ckpt.index
File already exists and is up-to-date: gpt2/774M/model.ckpt.meta
File already exists and is up-to-date: gpt2/774M/vocab.bpe
model loaded successfully.


In [9]:
import tiktoken, json
from torch.utils.data import Dataset
from transformers import TrainingArguments, Trainer

In [10]:
class JsonInstructionDataset(Dataset):
    def __init__(self, json_data, max_length=512,):
        self.encoding = tiktoken.get_encoding('gpt2')
        self.json_data = json_data
        self.max_length = max_length

    def __len__(self):
        return len(self.json_data)

    def __getitem__(self, idx):
        item = self.json_data[idx]

        text = f"### Instruction:\n{item['instruction']}\n\n### Input:\n{item['input']}\n\n### Response:\n{item['output']}"

        # tokenizing with tiktoken
        tokens = self.encoding.encode(text)

        # truncating if necessary
        if len(tokens) > self.max_length:
            tokens = tokens[:self.max_length]

        input_ids = tokens

        # padding if necessary
        if len(input_ids) < self.max_length:
            padding_length = self.max_length - len(input_ids)
            padding = [self.encoding.eot_token] * padding_length
            input_ids = input_ids + padding

        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'labels': torch.tensor(input_ids, dtype=torch.long)
        }

In [11]:
with open('/content/drive/MyDrive/llm_from_scratch/datasets/instruction_data.json', 'r') as f:
    json_data = json.load(f)

In [12]:
print(json_data[0])

{'instruction': 'Evaluate the following phrase by transforming it into the spelling given.', 'input': 'freind --> friend', 'output': 'The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".'}


In [13]:
dataset = JsonInstructionDataset(json_data)

In [14]:
dataset[0]

{'input_ids': tensor([21017, 46486,    25,   198,    36,  2100,  4985,   262,  1708,  9546,
           416, 25449,   340,   656,   262, 24993,  1813,    13,   198,   198,
         21017, 23412,    25,   198, 19503,   521, 14610,  1545,   198,   198,
         21017, 18261,    25,   198,   464, 24993,   286,   262,  1813,  9546,
           366, 19503,   521,     1,   318, 11491,    11,   262,  3376, 24993,
           318,   366,  6726,  1911, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50

In [15]:
from transformers import PretrainedConfig

class GPTConfig(PretrainedConfig):
    def __init__(self, **kwargs):
        # config
        self.vocab_size = 50257
        self.context_length = 1024
        self.emb_dim = 1280
        self.n_heads = 20
        self.n_layers = 36
        self.drop_rate = 0.1
        self.qkv_bias = True
        super().__init__(**kwargs)

# attaching config to model
model.config = GPTConfig()

In [16]:
import types

def hf_forward(self, input_ids=None, labels=None, **kwargs):

    batch_size, seq_len = input_ids.shape
    tok_embeds = self.tok_emb(input_ids)
    pos_embeds = self.pos_emb(torch.arange(seq_len, device=input_ids.device))
    x = tok_embeds + pos_embeds
    x = self.drop_emb(x)
    x = self.trf_blocks(x)
    x = self.final_norm(x)
    logits = self.out_head(x)

    # calculating loss if labels are present
    if labels is not None:
        loss_fct = torch.nn.CrossEntropyLoss()
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        return {'loss': loss, 'logits': logits}

    return {'logits': logits}

# applying the patch
model.forward = types.MethodType(hf_forward, model)
print("Fixed forward method")

✅ Fixed forward method (no recursion)


In [17]:
def prepare_inputs_for_generation(self, input_ids, **kwargs):
    return {"input_ids": input_ids, **kwargs}

if not hasattr(model, 'prepare_inputs_for_generation'):
    model.prepare_inputs_for_generation = prepare_inputs_for_generation.__get__(model, type(model))

In [18]:
from peft import LoraConfig, get_peft_model
def setup_lora_model(model, lora_r=16, lora_alpha=32):
    # using attention layers from my model
    target_modules = [
        "W_query",
        "W_key",
        "W_value",
        "out_proj",
    ]

    # LoRA configuration
    lora_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        target_modules=target_modules,
        lora_dropout=0.1,
        bias="none",
        task_type="CAUSAL_LM",
    )

    # applying LoRA
    model = get_peft_model(model, lora_config)

    return model

# applying LoRA to model
model = setup_lora_model(model)

In [19]:
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/llm_from_scratch/text_generation_model/instruction_model",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",
    remove_unused_columns=False,
    report_to="none"
)

In [20]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

In [21]:
os.environ["WANDB_DISABLED"] = "true"
print("Starting training...")
torch.cuda.empty_cache()

trainer.train()
print("Training completed!")

Starting training...


Step,Training Loss
50,3.268600
100,0.405600
150,0.350300
200,0.327900
250,0.330000
300,0.327500
350,0.317900
400,0.301000


Training completed!


In [31]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [80]:
def test_model(model, instruction, input_text="", max_new_tokens=1024):
    # Format the prompt like during training
    if input_text:
        prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"
    else:
        prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"

    # Tokenize
    encoding = tiktoken.get_encoding('gpt2')
    input_ids = encoding.encode(prompt)
    input_tensor = torch.tensor([input_ids]).to(device)  # Fixed: model.device instead of device

    # Manual generation (no .generate() method)
    model.eval()
    with torch.no_grad():
        generated = input_tensor

        for i in range(max_new_tokens):
            # Forward pass
            outputs = model(input_ids=generated)
            logits = outputs['logits']

            # Get last token logits
            next_token_logits = logits[0, -1, :]

            # Apply temperature and sample
            next_token_logits = next_token_logits / 0.7  # temperature
            probs = torch.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

            # Stop if EOT token is generated BEFORE appending
            if next_token.item() == encoding.eot_token:
                break

            # Append to generated sequence
            generated = torch.cat([generated, next_token.unsqueeze(0)], dim=1)

    # Decode
    response = encoding.decode(generated[0].tolist())
    # Extract only the response part (after "### Response:\n")
    response_text = response.split("### Response:\n")[-1]

    # Remove endoftext token if it exists and clean up
    response_text = response_text.replace('<|endoftext|>', '').strip()

    return response_text

# Test examples
print("🧪 Testing the fine-tuned model:\n")

# Example 1: General instruction
test1 = test_model(
    model,
    instruction="Explian what is the role of AI in computer science and medical?",
    input_text="",
)
print(f"Test 1 - Explanation:\n{test1}\n")

🧪 Testing the fine-tuned model:

Test 1 - Explanation:
The role of AI in computer science and medical sciences is rapidly advancing. AI can help render medical decisions faster, and it can also provide a more accurate diagnosis.

